In [1]:
#Imports
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
import glob
import pickle
from omegaconf import OmegaConf
from tqdm.auto import tqdm
from itertools import chain

sns.set_style("white")
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
cl100k = pd.read_pickle("/path/to/project/records/comp/fineweb2_hq/o200k_ratio/results.pkl")
domain_cols = cl100k.columns.drop("tok_name")
cl100k["concat"] = cl100k[domain_cols].apply(lambda row: list(chain.from_iterable(row)), axis=1)
cl100k = cl100k[["tok_name","concat"]]

In [3]:
X_train = pd.read_csv("/path/to/project/temp/finweb2_hq/X/mixture.csv")
X_train = X_train.sort_values(by="tok_name", key=lambda s: s.str.extract(r'(\d+)', expand=False).astype(int))
X_train = X_train.drop(columns=["Unnamed: 0"])

In [4]:
Y_train = pd.read_pickle("/path/to/project/records/comp/valid_txt_ratio_hf/fineweb2_hq/1gb_64k/results.pkl")
Y_train = Y_train.sort_values(by="tok_name", key=lambda s: s.str.extract(r'(\d+)', expand=False).astype(int)).reset_index(drop=True)
domain_cols = Y_train.columns.drop("tok_name")
Y_train["concat"] = Y_train[domain_cols].apply(lambda row: list(chain.from_iterable(row)), axis=1)
Y_train = Y_train[["tok_name","concat"]]

In [ ]:
cl100k.shape

In [ ]:
Y_train.shape

In [7]:
domains = Y_train.columns[1:]
def prepare_arr_y_from_df(df_y, cl100k):
    dist = []
    for idx in range(len(df_y)):  
        dist_per_tokenizer = []
        for domain in domains:    
            num = sum(df_y.loc[idx, domain])  
            den = sum(cl100k.loc[0, domain])  

            ratio = num / den if den != 0 else np.nan
            # ratio = 483087418 / num
            dist_per_tokenizer.append(ratio)
        dist.append(dist_per_tokenizer)
    return np.array(dist)

x_train = X_train[X_train.columns[1:]].values
y_train = prepare_arr_y_from_df(Y_train, cl100k)

In [ ]:
print(x_train.shape)
print(y_train.shape)

In [9]:
x_train, x_valid = x_train[:480, :], x_train[480:, :]
y_train, y_valid = y_train[:480, :], y_train[480:, :]

In [ ]:
KEY_METRICS = Y_train.columns[1:].tolist()
KEY_METRICS

In [ ]:
predictor=[]
hyper_params = {
    'task': 'train',
    'boosting_type': 'gbdt',
    'objective': 'regression',
    'metric': ['l1','l2'],
    "num_iterations": 1000, 
    'seed': 42,
    'learning_rate': 5e-2,
    "verbosity": -1,
}

np.random.seed(42)

print("=============================================================")

target = y_train[:,-1]
test_target = y_valid[:,-1]

gbm = lgb.LGBMRegressor(**hyper_params)

reg = gbm.fit(x_train, target,
    eval_set=[(x_valid, test_target)],
    eval_metric='l2', callbacks=[
])
r, p = spearmanr(reg.predict(x_valid), test_target)

mse = mean_squared_error(test_target, reg.predict(x_valid))

print("Correlation: {}".format(np.round(r*100, 2)))
print("MSE: {:.8f}".format(mse))
print("### Feature Importance")

for col, imp in zip(X_train.columns[1:], reg.feature_importances_):
    print(f"{col:10s} >>> {imp}")

predictor.append(reg)

In [ ]:
def calc_mape(y_true, y_pred, eps=1e-8):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100.0
print("===========================================================")
print("ALL Domain Predictor Performance")
test_target = y_valid[:,-1]
reg = predictor[0]

r, p = spearmanr(reg.predict(x_valid), test_target)
mse = mean_squared_error(test_target, reg.predict(x_valid))
mape = calc_mape(test_target, reg.predict(x_valid))
print("TOTAL Correlation: {:.3f}".format(r))
print("TOTAL MSE: {:.8f}".format(mse))
print("TOTAL MAPE: {:.8f}".format(mape))
print("===========================================================")


In [ ]:
# regression model works?
import matplotlib.pyplot as plt
import numpy as np

y_val_mean = y_valid[:,-1]
preds_mean = predictor[0].predict(x_valid)

plt.figure(figsize=(6,4))
plt.scatter(y_val_mean, preds_mean, alpha=0.7)
plt.plot([y_val_mean.min(), y_val_mean.max()],
         [y_val_mean.min(), y_val_mean.max()],
         'r--')
plt.xlabel("Actual Compression Ratio")
plt.ylabel("Predicted Compression Ratio")
plt.title("Predicted vs Actual", fontsize=16, fontweight="bold")
plt.show()

In [17]:
def pplot(optimal_data_mixture, which_metric):
    values = np.array(optimal_data_mixture)
    labels = np.array(X_train.columns[1:])
    mask = values > 0

    values = values[mask]
    labels = labels[mask]

    colors = plt.cm.tab20c(range(len(values)))

    plt.figure(figsize=(10, 8))
    wedges, texts, autotexts = plt.pie(
        values,
        labels=labels,
        autopct='%1.1f%%',
        startangle=140,
        counterclock=False,
        colors=colors,
        pctdistance=0.8,
        labeldistance=1.05
    )

    for text in texts:
        text.set_fontsize(10)
        text.set_fontweight("bold")
    for autotext in autotexts:
        autotext.set_fontsize(9)
        autotext.set_color("white")

    plt.title(
        f"Tokenizer Training Corpus Data Mixture {KEY_METRICS[which_metric]}",
        fontsize=14, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

def generate_train_group(groups, weights, precision=5):
    assert len(groups) == len(weights), "Length of groups and weights must be equal"
    
    def format_weight(weight):
        return f"{weight:.{precision}f}".rstrip('0').rstrip('.')
    
    output_group = [f"  {group}: {format_weight(num)}" 
                    for group, num in zip(groups, weights)]
    
    return "\n".join(output_group)

def generate_valid_group(groups):
    weights = [1.0] * len(groups)
    output_group = [f"  {group}: {num}" for group, num in zip(groups, weights)]
    return "\n".join(output_group)

def save_config(output_path, optimal_data_mixture):
    train_groups = list(X_train.columns[1:])
    valid_groups = list(X_train.columns[1:])
    weights = list(optimal_data_mixture)

    # get the train and valid group
    train_group = generate_train_group(train_groups, weights)
    valid_group = generate_valid_group(valid_groups)

    with open(output_path, "w", encoding="utf8") as f:
        f.write("train:\n")
        f.write(train_group)
        f.write("\n")
        f.write("valid:\n")
        f.write(valid_group)
        f.write("\n")
        
        # these are configurations for the model
        content = ""
        content += "\n" + "model_name: tinyllama_1M"
        # content += "\n" + "model_name: tinycoder_1M"
        content += "\n" + "total_devices: 1"
        content += "\n" + "num_of_devices: 1"
        content += "\n" + "global_batch_size: 512"
        content += "\n" + "micro_batch_size: 16"
        # 1001 instead of 1000 because wandb has the bug of not showing the last step
        content += "\n" + "max_step: 1001"
        
        # never save the model, just using the wandb log for regression fitting
        content += "\n" + "save_step_interval: 2000"
        content += "\n" + "eval_step_interval: 100"
        
        # constant learning rate for the small model
        content += "\n" + "learning_rate: 0.0004"
        content += "\n" + "min_lr: 0.0004"
        # the warmup step is 100
        content += "\n" + "warmup_steps: 100"
        f.write(content)

In [ ]:
np.random.seed(42)

prior_dist = [
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
    0.0526315789473684,
]

samples = np.random.dirichlet(prior_dist * 1, 50000000)
samples.shape

In [23]:
all_d_preds = []

pred = predictor[0].predict(samples)
all_d_preds.append(pred)

In [24]:
o = np.column_stack(all_d_preds)

In [ ]:
for i in range(len(KEY_METRICS)):
    clean = KEY_METRICS[i].split("-")[0]
    which_metric = i
    col = o[:, which_metric]
    for k in [1,8,32,64,256,512,1024]:
        topk_idx = np.argsort(col)[:k]
        topk_vals = col[topk_idx]

        print("Top-k indices:", topk_idx)
        print("Top-k values:", topk_vals)

        optimal_data_mixture = samples[topk_idx].mean(0)
        print("Optimal Data Mixture : ", optimal_data_mixture)
        pplot(optimal_data_mixture,which_metric)

        if k==1024:
            save_config(
                optimal_data_mixture=optimal_data_mixture,
                output_path=f"/path/to/project/configs/fineweb2_hq_trex_lang/{clean}_top{k}.yaml"
            )